# Mango V2: seasonal splits that keep repeated measurements together

Prepare four independent seasonal tasks from `NAnderson2020MendeleyMangoNIRData.csv`.
Each season is split into 80% training groups and 20% test groups with seed **42**.
Run all cells to generate the files in **`temp/mango_v2_grouped/`**.

### Why row splitting was unsafe

Repeated spectra share their fruit's dry matter (`DM`) reference. The source has
singletons as well as repeated-DM groups of two, four, six, and other sizes,
including measurements at different temperatures. Simply pairing every two
rows would miss some repeated measurements.

### Grouping assumption

V2 has no explicit fruit identifier. Within each season, **all rows with exactly
the same DM are assigned one conservative inferred fruit group**, regardless of
adjacency, source `Set`, temperature, or other metadata. This keeps the observed
repeat measurements together and may also group distinct fruits whose reported
DM happens to coincide. A zero-overlap check establishes separation of these
inferred groups; physical identity cannot be fully certified without fruit IDs.

The 80/20 ratio applies to groups. Groups contain different numbers of spectra,
so the spectrum counts need not be exactly 80/20. No target stratification,
preprocessing, outlier removal, or averaging is performed. Group IDs use first
appearance in the source; they are not ranked by DM magnitude.

Outputs use the benchmark column format: **101 raw spectral variables from
750 to 1050 nm, then `DM`**. Source-row and group IDs are recorded separately in
`sample_groups.csv`. The existing benchmark datasets and results are left in place.


## 1. Configuration and paths
Use the existing Python 3.13 environment (NumPy, pandas, and scikit-learn).

In [1]:
from pathlib import Path
import hashlib
import json
import platform

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import GroupShuffleSplit

# Run from the repository root or its temp folder.
SOURCE_FILENAME = "NAnderson2020MendeleyMangoNIRData.csv"
SEED = 42
TEST_GROUP_FRACTION = 0.20
SPECTRAL_COLUMNS = [str(w) for w in range(750, 1051, 3)]
MODEL_COLUMNS = SPECTRAL_COLUMNS + ["DM"]

source_candidates = [Path.cwd() / SOURCE_FILENAME,
                     Path.cwd() / "temp" / SOURCE_FILENAME]
SOURCE_PATH = next((p.resolve() for p in source_candidates if p.is_file()), None)
if SOURCE_PATH is None:
    raise FileNotFoundError(f"Place {SOURCE_FILENAME} in temp and run from temp or the repository root.")
ROOT = SOURCE_PATH.parent.parent
OUTPUT_ROOT = SOURCE_PATH.parent / "mango_v2_grouped"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_SHA256 = hashlib.sha256(SOURCE_PATH.read_bytes()).hexdigest()
print(f"Source: {SOURCE_PATH.name}")
print(f"Output: {OUTPUT_ROOT}")


Source: NAnderson2020MendeleyMangoNIRData.csv
Output: /mnt/c/Users/dario/OneDrive - Universidade do Algarve/WORK/Optica/Machine Learning/Deep Learning/Testes/0-NIRBENCH_DL_v0.1/temp/mango_v2_grouped


## 2. Load the source and separate the four seasons
Use `Season`, rather than calendar year, and retain every source observation.

In [2]:
source = pd.read_csv(SOURCE_PATH)
required = set(MODEL_COLUMNS + ["Season"])
missing = required - set(source.columns)
if missing:
    raise ValueError(f"Missing source columns: {sorted(missing)}")
if source["Season"].isna().any() or set(source["Season"].unique()) != {1, 2, 3, 4}:
    raise ValueError("Expected exactly four populated seasons, labelled 1 through 4.")
if not np.isfinite(source[MODEL_COLUMNS].to_numpy(dtype=float)).all():
    raise ValueError("Spectra and DM must be finite; no rows are silently dropped.")

# Split by Season rather than calendar year: each season spans two calendar years.
season_frames = {
    season: source.loc[source["Season"] == season].copy()
    for season in range(1, 5)
}
print(f"Source: {len(source):,} rows; {len(SPECTRAL_COLUMNS)} retained spectral variables (750–1050 nm).")
print(source.groupby("Season").size().rename("spectra").to_string())


Source: 11,691 rows; 101 retained spectral variables (750–1050 nm).
Season
1    3914
2    1363
3    4966
4    1448


## 3. Group repeated measurements and split
Assign each inferred fruit group wholly to training or test. Assertions check group separation, full row coverage, DM separation, and exact spectral duplicates.

In [3]:
def group_and_split(season_source, season):
    """Keep every equal-DM group within a season wholly in one partition."""
    # Factorization uses first appearance, not target magnitude or desired scores.
    # All exact DM matches are grouped, even if they might be distinct fruits.
    labels, _ = pd.factorize(season_source["DM"], sort=False)
    splitter = GroupShuffleSplit(
        n_splits=1, test_size=TEST_GROUP_FRACTION, random_state=SEED
    )
    train_positions, test_positions = next(
        splitter.split(season_source, groups=labels)
    )
    train_groups = set(labels[train_positions])
    test_groups = set(labels[test_positions])
    assert train_groups.isdisjoint(test_groups)
    assert set(train_positions).isdisjoint(test_positions)
    assert set(train_positions) | set(test_positions) == set(range(len(season_source)))

    train = season_source.iloc[train_positions][MODEL_COLUMNS].copy()
    test = season_source.iloc[test_positions][MODEL_COLUMNS].copy()
    shared_dm = set(train["DM"]) & set(test["DM"])
    assert not shared_dm, "A repeated reference value was split across partitions."

    # Exact row checks complement the inferred fruit grouping.
    train_spectra = set(map(tuple, train[SPECTRAL_COLUMNS].to_numpy()))
    test_spectra = set(map(tuple, test[SPECTRAL_COLUMNS].to_numpy()))
    shared_spectra = len(train_spectra & test_spectra)
    assert shared_spectra == 0, "Identical spectra occur in both partitions."

    partition = np.full(len(season_source), "train", dtype=object)
    partition[test_positions] = "test"
    metadata_columns = [c for c in ["Season", "Region", "Date", "Type", "Cultivar", "Pop", "Temp", "Set", "DM"]
                        if c in season_source.columns]
    membership = season_source[metadata_columns].copy()
    membership.insert(0, "source_row", season_source.index.to_numpy())
    membership.insert(1, "group_id", [f"S{season}_G{g:05d}" for g in labels])
    membership.insert(2, "partition", partition)
    assert membership.groupby("group_id")["partition"].nunique().max() == 1

    sizes = pd.Series(labels).value_counts()
    stats = {
        "task": f"{season + 21}-Mango_S{season}_dm",
        "season": season,
        "spectra": len(season_source),
        "groups": len(sizes),
        "train_groups": len(train_groups),
        "test_groups": len(test_groups),
        "train_rows": len(train),
        "test_rows": len(test),
        "test_row_fraction": len(test) / len(season_source),
        "shared_groups": len(train_groups & test_groups),
        "shared_dm_values": len(shared_dm),
        "identical_spectra_across_splits": shared_spectra,
        "seed": SEED,
        "test_group_fraction": TEST_GROUP_FRACTION,
    }
    return train, test, membership, stats


partitions = {}
summaries = []
for season, seasonal in season_frames.items():
    train, test, membership, stats = group_and_split(seasonal, season)
    partitions[season] = {"full": seasonal[MODEL_COLUMNS].copy(),
                          "train": train, "test": test, "membership": membership}
    summaries.append(stats)

summary = pd.DataFrame(summaries)
print(summary[["task", "spectra", "groups", "train_rows", "test_rows",
               "shared_groups", "shared_dm_values", "identical_spectra_across_splits"]].to_string(index=False))
print("All repeated-DM groups are confined to one partition; zero shared DM values and identical spectra.")


          task  spectra  groups  train_rows  test_rows  shared_groups  shared_dm_values  identical_spectra_across_splits
22-Mango_S1_dm     3914    1063        3132        782              0                 0                                0
23-Mango_S2_dm     1363     855        1086        277              0                 0                                0
24-Mango_S3_dm     4966    2027        3964       1002              0                 0                                0
25-Mango_S4_dm     1448     725        1158        290              0                 0                                0
All repeated-DM groups are confined to one partition; zero shared DM values and identical spectra.


## 4. Write the new datasets and provenance

Each `22-Mango_S1_dm` through `25-Mango_S4_dm` output directory contains:

- `data_full.csv`: the full seasonal spectral matrix and target.
- `data_train.csv` and `data_test.csv`: the new group-separated partitions.
- `sample_groups.csv`: source-row identity, inferred group, partition, and metadata;
  its row order matches `data_full.csv`, and filtering by partition matches the
  corresponding train/test CSV row order.

The output root also contains `split_summary.csv`, `manifest.csv`, and
`split_protocol.json`. Saved CSVs are read back and checked. Rerunning replaces
these generated files deterministically.


In [4]:
file_records = []
for season, data in partitions.items():
    task = f"{season + 21}-Mango_S{season}_dm"
    directory = OUTPUT_ROOT / task
    directory.mkdir(parents=True, exist_ok=True)
    # Repeated runs deterministically replace files only in this output folder.
    for split in ("full", "train", "test"):
        path = directory / f"data_{split}.csv"
        data[split].to_csv(path, index=False, lineterminator="\n")
        stored = pd.read_csv(path)
        pd.testing.assert_frame_equal(
            stored, data[split].reset_index(drop=True),
            check_dtype=False, check_exact=False, atol=1e-12, rtol=0,
        )
        file_records.append({
            "task": task, "split": split,
            "path": path.relative_to(OUTPUT_ROOT).as_posix(),
            "rows": len(stored), "spectral_features": len(SPECTRAL_COLUMNS),
            "target_column": "DM", "bytes": path.stat().st_size,
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        })
    membership_path = directory / "sample_groups.csv"
    data["membership"].to_csv(membership_path, index=False)
    # Recheck the saved partitions rather than relying only on in-memory results.
    saved_train = pd.read_csv(directory / "data_train.csv")
    saved_test = pd.read_csv(directory / "data_test.csv")
    assert set(saved_train["DM"]).isdisjoint(saved_test["DM"])
    assert set(map(tuple, saved_train[SPECTRAL_COLUMNS].to_numpy())).isdisjoint(
        set(map(tuple, saved_test[SPECTRAL_COLUMNS].to_numpy()))
    )
    stored_membership = pd.read_csv(membership_path)
    assert stored_membership.groupby("group_id")["partition"].nunique().max() == 1

summary.to_csv(OUTPUT_ROOT / "split_summary.csv", index=False)
pd.DataFrame(file_records).to_csv(OUTPUT_ROOT / "manifest.csv", index=False)
protocol = {
    "source_file": SOURCE_PATH.name, "source_sha256": SOURCE_SHA256,
    "source_rows": len(source), "seasons": [1, 2, 3, 4],
    "grouping": "All identical exact DM values within a season form one conservative inferred fruit group.",
    "identity_limit": "V2 contains no explicit fruit identifier. Equal DM is a proxy, not a verified physical identity.",
    "splitter": "sklearn.model_selection.GroupShuffleSplit",
    "random_state": SEED, "test_group_fraction": TEST_GROUP_FRACTION,
    "spectral_columns": SPECTRAL_COLUMNS, "target": "DM",
    "source_set_labels_used_for_split": False,
    "grouped_cv_required": "Use group_id from sample_groups.csv for future within-training validation folds.",
    "python": platform.python_version(), "numpy": np.__version__,
    "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
}
(OUTPUT_ROOT / "split_protocol.json").write_text(
    json.dumps(protocol, indent=2) + "\n", encoding="utf-8"
)
print("Wrote four seasonal datasets, four train/test pairs, group membership, and audit files.")
print(f"Output folder: {OUTPUT_ROOT}")


Wrote four seasonal datasets, four train/test pairs, group membership, and audit files.
Output folder: /mnt/c/Users/dario/OneDrive - Universidade do Algarve/WORK/Optica/Machine Learning/Deep Learning/Testes/0-NIRBENCH_DL_v0.1/temp/mango_v2_grouped


## Using these partitions later

The notebook only prepares new files in `temp/mango_v2_grouped/`. These splits
differ from those used for the existing benchmark results, so scores must be
recomputed when the new partitions are adopted.

Future five-fold validation on the new training data must also respect
`group_id` (for example, using `GroupKFold`). An ordinary row-based validation
split could still separate repeated spectra inside the training set.
